In [ ]:
import pandas as pd

In [ ]:
df_hist = pd.read_csv('../data/nfl_historico_con_elo.csv')

equipos = set(df_hist['home_team']).union(set(df_hist['away_team']))
elo_2025_final = {}

for equipo in equipos:
    df_equipo = df_hist[(df_hist['home_team'] == equipo) | (df_hist['away_team'] == equipo)]
    ultimo_juego = df_equipo.iloc[-1]

    if ultimo_juego['home_team'] == equipo:
        elo_2025_final[equipo] = ultimo_juego['elo_local_post']
    else:
        elo_2025_final[equipo] = ultimo_juego['elo_visita_post']

def regresion_media(rating):
    return (2/3 * rating) + (1/3 * 1500)

elo_2026_arranque = {eq: round(regresion_media(rat), 2) for eq, rat in elo_2025_final.items()}
print("✅ Ratings ajustados para el inicio de 2026 listos.")

✅ Ratings ajustados para el inicio de 2026 listos.


In [ ]:
df_2026 = pd.read_csv("../data/nfl_calendario_2026_clean.csv")

def calcular_probabilidad(rating_local, rating_visita, hfa=65):
    diferencia = (rating_local + hfa) - rating_visita
    return 1 / (1 + 10 ** (-diferencia / 400))

predicciones_2026 = []

for index, row in df_2026.iterrows():
    eq_local = row['home_team']
    eq_visita = row['away_team']
    
    elo_local = elo_2026_arranque.get(eq_local, 1500)
    elo_visita = elo_2026_arranque.get(eq_visita, 1500)
    
    prob_local = calcular_probabilidad(elo_local, elo_visita)
    
    predicciones_2026.append({
        'season': row['season'],
        'week': row['week'],
        'home_team': eq_local,
        'away_team': eq_visita,
        'prob_local': round(prob_local, 4),
        'prob_visita': round(1 - prob_local, 4)
    })

df_predicciones = pd.DataFrame(predicciones_2026)
print(f"🏈 {len(df_predicciones)} partidos de 2026 simulados.")

🏈 272 partidos de 2026 simulados.


In [ ]:
df_predicciones.head(20)

,season,week,home_team,away_team,prob_local,prob_visita
0,2026,1,SEA,NE,0.6575,0.3425
1,2026,1,LA,SF,0.6142,0.3858
2,2026,1,CAR,CHI,0.5205,0.4795
3,2026,1,CIN,TB,0.5796,0.4204
4,2026,1,DET,NO,0.7069,0.2931
5,2026,1,HOU,BUF,0.5658,0.4342
6,2026,1,IND,BAL,0.5307,0.4693
7,2026,1,JAX,CLE,0.6881,0.3119
8,2026,1,PIT,ATL,0.6386,0.3614
9,2026,1,TEN,NYJ,0.5774,0.4226


In [ ]:
# Mapeo de la estructura de la NFL
nfl_estructura = {
    # AFC EAST
    'BUF': {'conferencia': 'AFC', 'division': 'AFC East'},
    'MIA': {'conferencia': 'AFC', 'division': 'AFC East'},
    'NE':  {'conferencia': 'AFC', 'division': 'AFC East'},
    'NYJ': {'conferencia': 'AFC', 'division': 'AFC East'},
    
    # AFC NORTH
    'BAL': {'conferencia': 'AFC', 'division': 'AFC North'},
    'CIN': {'conferencia': 'AFC', 'division': 'AFC North'},
    'CLE': {'conferencia': 'AFC', 'division': 'AFC North'},
    'PIT': {'conferencia': 'AFC', 'division': 'AFC North'},
    
    # AFC SOUTH
    'HOU': {'conferencia': 'AFC', 'division': 'AFC South'},
    'IND': {'conferencia': 'AFC', 'division': 'AFC South'},
    'JAX': {'conferencia': 'AFC', 'division': 'AFC South'},
    'TEN': {'conferencia': 'AFC', 'division': 'AFC South'},
    
    # AFC WEST
    'DEN': {'conferencia': 'AFC', 'division': 'AFC West'},
    'KC':  {'conferencia': 'AFC', 'division': 'AFC West'},
    'LV':  {'conferencia': 'AFC', 'division': 'AFC West'},
    'LAC': {'conferencia': 'AFC', 'division': 'AFC West'},
    
    # NFC EAST
    'DAL': {'conferencia': 'NFC', 'division': 'NFC East'},
    'NYG': {'conferencia': 'NFC', 'division': 'NFC East'},
    'PHI': {'conferencia': 'NFC', 'division': 'NFC East'},
    'WAS': {'conferencia': 'NFC', 'division': 'NFC East'},
    
    # NFC NORTH
    'CHI': {'conferencia': 'NFC', 'division': 'NFC North'},
    'DET': {'conferencia': 'NFC', 'division': 'NFC North'},
    'GB':  {'conferencia': 'NFC', 'division': 'NFC North'},
    'MIN': {'conferencia': 'NFC', 'division': 'NFC North'},
    
    # NFC SOUTH
    'ATL': {'conferencia': 'NFC', 'division': 'NFC South'},
    'CAR': {'conferencia': 'NFC', 'division': 'NFC South'},
    'NO':  {'conferencia': 'NFC', 'division': 'NFC South'},
    'TB':  {'conferencia': 'NFC', 'division': 'NFC South'},
    
    # NFC WEST
    'ARI': {'conferencia': 'NFC', 'division': 'NFC West'},
    'LA':  {'conferencia': 'NFC', 'division': 'NFC West'}, 
    'SF':  {'conferencia': 'NFC', 'division': 'NFC West'},
    'SEA': {'conferencia': 'NFC', 'division': 'NFC West'}
}

In [ ]:
victorias_esperadas = {equipo: 0.0 for equipo in equipos}

# Sumar las probabilidades de cada partido para obtener el número esperado de victorias[cite: 2]
for index, row in df_predicciones.iterrows():
    victorias_esperadas[row['home_team']] += row['prob_local']
    victorias_esperadas[row['away_team']] += row['prob_visita']

# Crear tabla de posiciones proyectada[cite: 2]
df_standings = pd.DataFrame([
    {
        'Equipo': eq, 
        'Victorias_Proyectadas': round(wins, 1), 
        'Derrotas_Proyectadas': round(17 - wins, 1)
    }
    for eq, wins in victorias_esperadas.items()
]).sort_values(by='Victorias_Proyectadas', ascending=False)

# Exportar las dos tablas requeridas para la interfaz[cite: 2]
df_predicciones.to_csv("../data/nfl_predicciones_2026.csv", index=False)

df_standings['Conferencia'] = df_standings['Equipo'].map(lambda x: nfl_estructura[x]['conferencia'])
df_standings['Division'] = df_standings['Equipo'].map(lambda x: nfl_estructura[x]['division'])

# 2. Reordenar las columnas para que se vea más limpio
df_standings = df_standings[['Equipo', 'Conferencia', 'Division', 'Victorias_Proyectadas', 'Derrotas_Proyectadas']]

# 3. Guardar el archivo definitivo
df_standings.to_csv("../data/nfl_standings_2026.csv", index=False)
display(df_standings.head())

,Equipo,Conferencia,Division,Victorias_Proyectadas,Derrotas_Proyectadas
26,SEA,NFC,NFC West,10.6,6.4
28,PHI,NFC,NFC East,10.2,6.8
25,DET,NFC,NFC North,9.9,7.1
12,BUF,AFC,AFC East,9.8,7.2
4,DEN,AFC,AFC West,9.8,7.2


In [ ]:
df_standings.head(30)

,Equipo,Conferencia,Division,Victorias_Proyectadas,Derrotas_Proyectadas
26,SEA,NFC,NFC West,10.6,6.4
28,PHI,NFC,NFC East,10.2,6.8
25,DET,NFC,NFC North,9.9,7.1
12,BUF,AFC,AFC East,9.8,7.2
4,DEN,AFC,AFC West,9.8,7.2
21,LA,NFC,NFC West,9.7,7.3
3,HOU,AFC,AFC South,9.7,7.3
29,SF,NFC,NFC West,9.4,7.6
27,MIN,NFC,NFC North,9.3,7.7
24,BAL,AFC,AFC North,9.2,7.8
